# 06 — Hybrid CNN-BiLSTM: Training & Evaluation

This is the **primary model** of the project — the best-performing architecture achieving **96.20% test accuracy**.

### Architecture Intuition

Standard CNNs extract local texture features from each region of the image independently.  
The hybrid approach adds a **Bidirectional LSTM** on top:

1. CNN extracts a 2D feature map from the cell image
2. Each row of the feature map is treated as a **timestep**
3. BiLSTM reads these timesteps in both directions, capturing **spatial dependencies** across the cell

This is especially useful for malaria detection because parasite morphology involves patterns that span across cell regions — not just local textures.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger

from models import build_hybrid_cnn_lstm
from preprocessing import load_splits, get_data_generators
from evaluate import plot_training_curves, evaluate_model

print(f'TensorFlow: {tf.__version__}')
gpu = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpu[0].name if gpu else "None (CPU mode)"}')

## 1. Load Preprocessed Data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits('../data')
train_gen, val_gen = get_data_generators(X_train, y_train, X_val, y_val)

print(f'Training samples  : {len(X_train):,}')
print(f'Validation samples: {len(X_val):,}')
print(f'Test samples      : {len(X_test):,}')

# Class distribution in test set
unique, cnts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, cnts):
    label = 'Parasitized' if u == 1 else 'Uninfected'
    print(f'  Test {label}: {c:,}')

## 2. Build Hybrid Model

In [ ]:
model = build_hybrid_cnn_lstm()

model.compile(
    optimizer=Adam(learning_rate=5e-4),   # Slightly lower LR for LSTM stability
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

model.summary()
print(f'\nTotal trainable parameters: {model.count_params():,}')

## 3. Visualise Architecture

The data flow through the hybrid model:

In [ ]:
# Architecture summary diagram
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

stages = [
    ('Input',          '128 × 128 × 3',          '#58a6ff'),
    ('Conv Block 1',   '64 × 64 × 32 (+ BN)',    '#388bfd'),
    ('Conv Block 2',   '32 × 32 × 64 (+ BN)',    '#1f6feb'),
    ('Conv Block 3',   '16 × 16 × 128 (+ BN)',   '#1158c7'),
    ('Conv Block 4',   '8 × 8 × 256 (+ BN)',     '#0d419d'),
    ('Reshape',        '8 timesteps × 2048 dims', '#3fb950'),
    ('BiLSTM Layer 1', '8 × 256 (bidirectional)', '#2ea043'),
    ('BiLSTM Layer 2', '128 (final state)',        '#238636'),
    ('Dense + Dropout','128 units, drop 0.5',     '#e3b341'),
    ('Output',         '1 unit (sigmoid)',         '#f85149'),
]

y_pos = 0.95
for name, detail, color in stages:
    ax.text(0.05, y_pos, f'▶ {name}', fontsize=10, fontweight='bold',
            color=color, transform=ax.transAxes)
    ax.text(0.45, y_pos, detail, fontsize=9, color='#8b949e',
            transform=ax.transAxes)
    y_pos -= 0.09
    if y_pos > 0.05:
        ax.annotate('', xy=(0.13, y_pos + 0.065), xytext=(0.13, y_pos + 0.02),
                    xycoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#30363d', lw=1.5))

ax.set_title('Hybrid CNN-BiLSTM Architecture', fontsize=13, fontweight='bold',
             pad=10, color='#e6edf3')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.savefig('../results/hybrid_architecture.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

## 4. Train

In [ ]:
EPOCHS     = 50
BATCH_SIZE = 32

callbacks = [
    ModelCheckpoint(
        '../results/saved_models/hybrid_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
    CSVLogger('../results/training_logs/hybrid_training_log.csv')
]

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    validation_data=val_gen,
    validation_steps=len(X_val) // BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print('\nTraining complete.')

## 5. Training Curves

In [ ]:
plot_training_curves(history, model_name='hybrid')

## 6. Test Set Evaluation & Confusion Matrix

In [ ]:
metrics = evaluate_model(model, X_test, y_test, model_name='hybrid')

print('\n--- Final Metrics (Hybrid CNN-BiLSTM) ---')
for k, v in metrics.items():
    if k != 'model':
        print(f'  {k:12s}: {v:.4f}  ({v*100:.2f}%)')

## 7. Inference on Individual Images

Test the saved model on a few individual images from the test set.

In [ ]:
CLASS_NAMES = {0: 'Uninfected', 1: 'Parasitized'}
COLORS_MAP  = {'Parasitized': '#f85149', 'Uninfected': '#3fb950'}

# Pick 8 random test images
rng     = np.random.default_rng(99)
indices = rng.choice(len(X_test), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, idx in enumerate(indices):
    img    = X_test[idx]
    true   = CLASS_NAMES[y_test[idx]]
    prob   = float(model.predict(img[np.newaxis], verbose=0)[0][0])
    pred   = CLASS_NAMES[int(prob >= 0.5)]
    conf   = prob if int(prob >= 0.5) == 1 else (1 - prob)
    correct = pred == true

    axes[i].imshow(img)
    axes[i].axis('off')
    color = COLORS_MAP[pred]
    border_color = '#3fb950' if correct else '#f85149'
    for spine in axes[i].spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    axes[i].set_title(
        f'True: {true}\nPred: {pred} ({conf*100:.1f}%)',
        fontsize=8.5, color=color
    )

fig.suptitle('Hybrid CNN-BiLSTM — Sample Predictions\n(Green border = correct, Red = incorrect)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/hybrid_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Metric | Value |
|---|---|
| Test Accuracy | **96.20%** |
| Precision | 0.963 |
| Recall | 0.962 |
| F1-Score | 0.962 |
| AUC-ROC | ~0.99 |

The hybrid model slightly outperforms standalone CNNs on complex parasite morphologies by leveraging sequential spatial reasoning via the BiLSTM layers.

→ Proceed to the **Flask web app** in `app/app.py` to test the deployed model.